# Job Scraping Analysis Notebook
**Name**: Mayenmein Terence Sama Aloah Jr<br>
**Date**: October 2025<br>
**Project**: SkillHub Job Data Collection
## Introduction
This notebook demonstrates the functionality of the JobScraper class for collecting job posting data from the Found.dev API and storing it directly in PostgreSQL database. The implementation focuses on batch processing, database efficiency, and progress tracking.

## 1. Import and Setup
Let's start by importing the necessary modules and setting up our environment.

In [1]:
import sys
import os
import pandas as pd
from datetime import datetime
from pathlib import Path
import psycopg2
# Add the src directory to the path to import our custom module
sys.path.append('..')

# Import the JobScraper class
from src.scraping.scrape_jobs import JobScraper
from src.scraping.scrape_cam import JobDataAPIStreamingScraper
from src.database.create_database import DatabaseCreator

print("✅ Imports completed successfully!")
print(f"📅 Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Imports completed successfully!
📅 Analysis date: 2026-07-21 22:20:51


## 3. Test Database Connection and Schema
Verify that we can connect to the database and check the table structure.

In [2]:
database_manager = DatabaseCreator()
database_manager.run()

Connecting to PostgreSQL at localhost:5432...
Connected to PostgreSQL as postgres
Database 'data_science_job_market_db' already exists
Successfully connected to 'data_science_job_market_db' with regular user credentials


In [3]:
datajobs_scraper = JobDataAPIStreamingScraper()

Error creating tables: relation "datajobscompanies" already exists



In [4]:
total = datajobs_scraper.scrape_by_country('CM')
datajobs_scraper.get_database_stats()

Streaming jobs from JobDataAPI for country: CM


ModuleNotFoundError: No module named 'brotli'

## 2. Initialize the Job Scraper
Create an instance of the JobScraper class with custom configuration.

In [5]:
scraper = JobScraper()

print(f"Scraper initialized successfully!")
print(f"API endpoint: {scraper.BASE_URL}")
print(f"Database: {scraper.db_config['dbname']} on {scraper.db_config['host']}")


Database tables created successfully from SQL schema
Scraper initialized successfully!
API endpoint: https://api.found.dev/api/open/jobs
Database: data_science_job_market_db on localhost


## 4. Test Single Page Fetch
Before running full batch scraping, let's test fetching a single page to understand the data structure.

In [6]:
def test_single_fetch():
    """Test fetching a single page of job data"""
    print("🔍 Testing single page fetch...")
    
    try:
        # Fetch first page
        data = scraper.fetch_jobs(page=1, skill="Data Science", ai=True)
        jobs = data.get("jobs", [])
        
        print(f"Jobs found on page 1: {len(jobs)}")
        
        if jobs:
            # Process the jobs
            companies, jobs, skill_details = scraper.process_job_data(jobs[:2])  # Process first 2 jobs as sample
            print(f"Processed jobs sample: {len(jobs)}")
            print(f"Processed companies sample: {len(companies)}")
            print(f"Processed skill details sample: {len(skill_details)}")
            
            # Display sample data
            if companies:
                company_df = pd.DataFrame(companies)
            if skill_details:
                skill_df = pd.DataFrame(skill_details)
            if jobs:
                job_df = pd.DataFrame(jobs)
            if not job_df.empty and not company_df.empty and not skill_df.empty:
                display(company_df.head(), skill_df.head(), job_df.head())
        return len(jobs)
        
    except Exception as e:
        print(f"Error during test fetch: {e}")
        return 0

# Run the test
jobs_count = test_single_fetch()
print(f"\n✅ Single page test completed. Found {jobs_count} jobs.")

🔍 Testing single page fetch...
Jobs found on page 1: 100
Processed jobs sample: 2
Processed companies sample: 2
Processed skill details sample: 38


,slug,name,description,description_gpt,description_premium,description_linkedin,description_combined_gpt,description_perplexity,logo,url,...,public_vs_private,company_sector,linkedin_tags,key_products,using_ai,ai_products_gpt,mission_values,jobs_count,jobs_ai_count,last_job_source
0,nelnet,Nelnet,,Nelnet Business Services provides payment tech...,,,,,uploads/VL64JGMLT7DE.gif,https://nelnet.dejobs.org,...,,,,,,,,60,52,
1,ntt-america-inc,"NTT America, Inc.",,NTT DATA is a global business and technology s...,,,,,logos/company-nologo.svg,,...,,,,,,,,415,295,


,0,1,2
0,nelnet-senior-data-scientist-qnxw,Machine Learning,technical
1,nelnet-senior-data-scientist-qnxw,Testing,technical
2,nelnet-senior-data-scientist-qnxw,Python,technical
3,nelnet-senior-data-scientist-qnxw,LLMs,technical
4,nelnet-senior-data-scientist-qnxw,Cloud,technical


,slug,company_slug,title,description,premium,ai,status,created_at,published,pin_until,...,salary_currency,salary_period,benefits,beneficial,experience,ideal_candidate,qualifications,schema,force_status,same_as
0,nelnet-senior-data-scientist-qnxw,nelnet,Senior Data Scientist,,False,True,1,None,2026-07-21T19:45:45.629111Z,None,...,USD,annual,,,,,,,,0
1,ntt-america-inc-ai-consultant-products-jter,ntt-america-inc,AI Consultant - Products,,False,True,1,None,2026-07-21T19:45:43.419533Z,None,...,USD,annual,,,,,,,,0



✅ Single page test completed. Found 2 jobs.


## 5. Run Small Batch Scraping
Now let's run a small batch scraping operation to demonstrate the functionality with database storage.

In [7]:
def run_small_batch_scraping():
    """Run scraping with a small batch size for demonstration"""
    print("Starting small batch scraping...")
    print("Jobs will be saved directly to PostgreSQL database")
    
    # Run with small batch size for quick demonstration
    total_jobs = scraper.scrape_in_batches(
        skill="Data Science",
        pages_per_batch=3,
        ai=True,
        delay=1,
        max_batches=2
    )
    
    print(f"\nSmall batch scraping completed!")
    print(f"Total jobs collected: {total_jobs}")
    
    return total_jobs

# Execute small batch scraping
small_batch_total = run_small_batch_scraping()

Starting small batch scraping...
Jobs will be saved directly to PostgreSQL database
Scraping Data Science jobs...


Overall Progress: 100jobs [01:01,  1.61jobs/s, total=100, batches=1]

Batch 1 complete: +100 jobs (Total: 100)


Overall Progress: 200jobs [02:00,  1.67jobs/s, total=200, batches=2]

Batch 2 complete: +100 jobs (Total: 200)

Reached maximum batch limit: 2

Small batch scraping completed!
Total jobs collected: 200


## 6. Data Quality Check via Database
Perform data quality checks by querying the database.

In [ ]:
def check_database_quality():
    """Run essential data quality checks."""

    print("🔍 Running data quality checks...")

    try:
        with psycopg2.connect(**scraper.db_config) as conn:

            quality = pd.read_sql("""
                SELECT
                    COUNT(*) AS total_jobs,

                    COUNT(*) FILTER (
                        WHERE job_slug IS NULL OR TRIM(job_slug) = ''
                    ) AS missing_job_slug,

                    COUNT(*) FILTER (
                        WHERE title IS NULL OR TRIM(title) = ''
                    ) AS missing_title,

                    COUNT(*) FILTER (
                        WHERE company_slug IS NULL OR TRIM(company_slug) = ''
                    ) AS missing_company,

                    COUNT(*) FILTER (
                        WHERE country IS NULL OR TRIM(country) = ''
                    ) AS missing_country,

                    COUNT(*) FILTER (
                        WHERE published IS NULL
                    ) AS missing_dates,

                    COUNT(*) FILTER (
                        WHERE salary_min < 0
                           OR salary_max < 0
                           OR salary_min > salary_max
                    ) AS invalid_salaries,

                    COUNT(*) FILTER (
                        WHERE published > CURRENT_TIMESTAMP
                    ) AS future_dates

                FROM jobs
            """, conn)

            duplicates = pd.read_sql("""
                SELECT COUNT(*) AS duplicate_job_records
                FROM (
                    SELECT job_slug
                    FROM jobs
                    GROUP BY job_slug
                    HAVING COUNT(*) > 1
                ) duplicates
            """, conn)

            orphaned_jobs = pd.read_sql("""
                SELECT COUNT(*) AS orphaned_jobs
                FROM jobs j
                LEFT JOIN companies c
                    ON j.company_slug = c.slug
                WHERE c.slug IS NULL
            """, conn)

            orphaned_skills = pd.read_sql("""
                SELECT COUNT(*) AS orphaned_skill_records
                FROM job_skills_detail s
                LEFT JOIN jobs j
                    ON s.job_slug = j.job_slug
                WHERE j.job_slug IS NULL
            """, conn)

            print("\n📊 DATA QUALITY REPORT")
            print("=" * 50)
            print(quality.to_string(index=False))
            print(duplicates.to_string(index=False))
            print(orphaned_jobs.to_string(index=False))
            print(orphaned_skills.to_string(index=False))

            return {
                "quality": quality,
                "duplicates": duplicates,
                "orphaned_jobs": orphaned_jobs,
                "orphaned_skills": orphaned_skills
            }

    except Exception as e:
        print(f"❌ Quality check failed: {e}")
        return None


# Run quality checks
quality_report = check_database_quality()

## 7. Full-Scale Scraping
For comprehensive data collection, run the full scraping process.

In [ ]:
def run_comprehensive_scraping():
    """Run comprehensive job scraping (optional - may take time)"""
    print("🔍 Starting comprehensive scraping...")
    print("⚠️  This may take 30-60 minutes depending on job volume")
    
    # You can adjust these parameters based on your needs
    total_jobs = scraper.scrape_in_batches(
        skill="Data Science",
        pages_per_batch=20,  # Larger batches for efficiency
        ai=True,
        delay=1,
        max_batches=None  # No limit, stops when no more jobs
    )
    
    print(f"\n🏁 Comprehensive scraping completed!")
    print(f"📈 Total jobs collected: {total_jobs}")
    
    return total_jobs

comprehensive_total = run_comprehensive_scraping()


🔍 Starting comprehensive scraping...
⚠️  This may take 30-60 minutes depending on job volume
Scraping Data Science jobs...


Overall Progress: 90jobs [11:21,  7.57s/jobs, total=90, batches=1]

Batch 1 complete: +90 jobs (Total: 90)


Overall Progress: 180jobs [29:41, 10.31s/jobs, total=180, batches=2]

Batch 2 complete: +90 jobs (Total: 180)


...

Overall Progress: 1080jobs [1:59:17,  4.18s/jobs, total=1080, batches=11]

Batch 11 complete: +100 jobs (Total: 1080)


Overall Progress: 1179jobs [2:03:29,  3.68s/jobs, total=1179, batches=12]

Batch 12 complete: +99 jobs (Total: 1179)



Error on page 259: ('Connection broken: IncompleteRead(4524 bytes read, 5716 more expected)', IncompleteRead(4524 bytes read, 5716 more expected))

Error on page 260: HTTPSConnectionPool(host='api.found.dev', port=443): Max retries exceeded with url: /api/open/jobs?page=260&skill=Data+Science&ai=true (Caused by NameResolutionError("HTTPSConnection(host='api.found.dev', port=443): Failed to resolve 'api.found.dev' ([Errno 11001] getaddrinfo failed)"))


In [ ]:
print("\n🎉 Notebook execution complete! All data is now stored in PostgreSQL database.")